# 01. INGESTA BRONZE - BUK COLOMBIA API
**Proyecto:** Datamart Gestión de Nómina Maaji
Extracción automatizada y persistencia Delta Lake / Parquet.


In [ ]:
# Databricks notebook source
# MAGIC %pip install requests pandas openpyxl


In [ ]:
import os
import requests
import json
import pandas as pd
from datetime import datetime

# En Databricks, usar Secret Scope:
# BUK_TOKEN = dbutils.secrets.get(scope="datamart-secrets", key="buk-api-token")
BUK_TOKEN = os.getenv("BUK_API_TOKEN", "EMVvA6ppoVTtRdoXgqu8JSWj")
BUK_TENANT = os.getenv("BUK_TENANT", "maaji")
BASE_URL = f"https://{BUK_TENANT}.buk.co/api/v1/colombia"

headers = {"auth_token": BUK_TOKEN, "Accept": "application/json"}
print("Conectando a Buk Colombia API...")


In [ ]:
def extract_paginated(endpoint, page_size=100):
    all_data = []
    page = 1
    while True:
        url = f"{BASE_URL}/{endpoint}?page={page}&page_size={page_size}"
        resp = requests.get(url, headers=headers, timeout=20)
        if resp.status_code != 200:
            break
        body = resp.json()
        items = body.get("data", [])
        if not items:
            break
        all_data.extend(items)
        if not body.get("pagination", {}).get("next"):
            break
        page += 1
    return all_data

# Extracción de entidades principales
df_companies = pd.DataFrame(requests.get(f"{BASE_URL}/companies", headers=headers).json().get("data", []))
df_areas = pd.DataFrame(extract_paginated("areas"))
df_roles = pd.DataFrame(extract_paginated("roles"))
df_locations = pd.DataFrame(extract_paginated("locations"))
df_periods = pd.DataFrame(extract_paginated("process_periods"))
df_items = pd.DataFrame(extract_paginated("items"))
df_employees = pd.DataFrame(extract_paginated("employees"))

print(f"Colaboradores extraídos: {len(df_employees)}")
print(f"Conceptos de nómina extraídos: {len(df_items)}")


In [ ]:
# Guardado en Delta Lake (Databricks)
# spark.createDataFrame(df_employees).write.format("delta").mode("overwrite").saveAsTable("talent_bronze.buk_employees")
print("Ingesta Bronze completada.")
